# Paper tables, figures, and reproducibility package

**Public role:** Core table/figure generation and final packaging notebook.

**Relation to the manuscript:** Generates the compact tables, supplementary evidence, and plot-only reference outputs used by the current article version.

**Execution status of this curated copy:** Stored outputs and execution counters were removed to prevent disclosure of local paths, embedded dataset images, stale errors, and large widget state. The code was not re-executed after this mechanical curation. Reference outputs are distributed separately under `results/`.

**Path configuration:** Set `AQPR_PROJECT_ROOT` to the working directory containing `project_data/`, `models/`, and `results/`. Set `AQPR_DATA_ROOT` to the external dataset root. The default values target a Google Drive/Colab workflow and can be changed without editing the notebook logic.

**Original source notebook:** `16_ict_express_revision_tables_figures_and_response_package_v36.ipynb`


# Notebook 16 — ICT Express v3.6 Final Minor-Revision Tables, Figures, and Response Package

Consolidates Notebook 13, Notebook 14 v3.6, and Notebook 15 v3.6 outputs for the final reviewer-driven ICT Express manuscript package.

In [ ]:
import os
# Cell 1: Setup, Drive mount, constants, and output folders — v3.6

from pathlib import Path
import os, re, json, shutil, warnings
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

NOTEBOOK_NAME = "16_ict_express_revision_tables_figures_and_response_package_v36"
PRIMARY_MODEL_NAME = "EfficientNetB0"
SECONDARY_MODEL_NAME = "ConvNeXtTiny"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    COLAB_DRIVE_MOUNTED = True
except ModuleNotFoundError:
    COLAB_DRIVE_MOUNTED = False
    print("google.colab is not available. Running outside Colab or with local runtime.")
except Exception as exc:
    COLAB_DRIVE_MOUNTED = False
    print(f"Google Drive mount failed or skipped: {type(exc).__name__}: {exc}")

PROJECT_ROOT = Path(os.environ.get("AQPR_PROJECT_ROOT", "/content/drive/MyDrive/AQPR_Glaucoma"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/mnt/data")

RESULTS_ROOT = PROJECT_ROOT / "results"
RESULTS_16_DIR = RESULTS_ROOT / "16_ict_express_revision_tables_figures_and_response_package_v36"

TABLE_DIR = RESULTS_16_DIR / "tables"
SUPP_TABLE_DIR = RESULTS_16_DIR / "supplementary_tables"
FIGURE_DIR = RESULTS_16_DIR / "figures"
TEXT_DIR = RESULTS_16_DIR / "text_fragments"
LATEX_DIR = RESULTS_16_DIR / "latex_tables"
AUDIT_DIR = RESULTS_16_DIR / "audit"
PACKAGE_DIR = RESULTS_16_DIR / "paper_revision_package"

for d in [RESULTS_16_DIR, TABLE_DIR, SUPP_TABLE_DIR, FIGURE_DIR, TEXT_DIR, LATEX_DIR, AUDIT_DIR, PACKAGE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 1",
    "status": "PASS",
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "colab_drive_mounted": bool(COLAB_DRIVE_MOUNTED),
    "project_root": str(PROJECT_ROOT),
    "results_16_dir": str(RESULTS_16_DIR),
    "purpose": "Consolidate v3.6 minor-revision evidence: endpoint definitions, matched-random robustness, secondary attribution check, outside-OD forensics, and proxy CDR baseline."
}
with open(RESULTS_16_DIR / "notebook16_v36_cell01_setup_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:
# Cell 2: Utilities and upstream discovery — v3.6

def exists_nonempty(path_like):
    if path_like is None:
        return False
    p = Path(path_like)
    return bool(p.exists() and p.is_file() and p.stat().st_size > 0)

def dir_exists(path_like):
    if path_like is None:
        return False
    p = Path(path_like)
    return bool(p.exists() and p.is_dir())

def first_existing_dir(candidates):
    for p in candidates:
        if dir_exists(p):
            return Path(p)
    return None

def first_dir_with_file(candidates, relative_file):
    for p in candidates:
        if dir_exists(p) and exists_nonempty(Path(p) / relative_file):
            return Path(p)
    return None

def first_existing_file(candidates):
    for p in candidates:
        if exists_nonempty(p):
            return Path(p)
    return None

def safe_read_csv(path_like):
    if path_like is None or not exists_nonempty(path_like):
        return pd.DataFrame()
    try:
        return pd.read_csv(path_like, low_memory=False)
    except Exception as exc:
        print(f"Could not read CSV {path_like}: {type(exc).__name__}: {exc}")
        return pd.DataFrame()

def safe_read_json(path_like):
    if path_like is None or not exists_nonempty(path_like):
        return {}
    try:
        with open(path_like, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        print(f"Could not read JSON {path_like}: {type(exc).__name__}: {exc}")
        return {}

def copy_if_exists(src, dst_dir, new_name=None):
    if src is None or not Path(src).exists():
        return ""
    src = Path(src)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / (new_name if new_name else src.name)
    try:
        shutil.copy2(src, dst)
        return str(dst)
    except Exception as exc:
        print(f"Could not copy {src}: {type(exc).__name__}: {exc}")
        return ""

def save_csv_md(df, csv_path, md_path=None, max_rows=120):
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(csv_path, index=False)
    if md_path is not None:
        md_path = Path(md_path)
        md_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            df.head(max_rows).to_markdown(md_path, index=False)
        except Exception:
            md_path.write_text(df.head(max_rows).to_string(index=False), encoding="utf-8")

def write_latex_table(df, path, caption, label, max_rows=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if df is None or df.empty:
        path.write_text("% Table not generated because source dataframe was empty.\n", encoding="utf-8")
        return False
    tmp = df.copy()
    if max_rows is not None:
        tmp = tmp.head(max_rows)
    try:
        latex = tmp.to_latex(index=False, escape=True, caption=caption, label=label)
    except Exception as exc:
        latex = f"% Failed to generate LaTeX table: {type(exc).__name__}: {exc}\n"
    path.write_text(latex, encoding="utf-8")
    return True

def fmt_ci(point, low=None, high=None, decimals=3):
    try:
        if pd.isna(point):
            return "NA"
        if low is not None and high is not None and pd.notna(low) and pd.notna(high):
            return f"{float(point):.{decimals}f} [{float(low):.{decimals}f}, {float(high):.{decimals}f}]"
        return f"{float(point):.{decimals}f}"
    except Exception:
        return "NA"

def select_cols(df, cols):
    return [c for c in cols if c in df.columns]

# Upstream directories.
N13_DIR = first_existing_dir([
    RESULTS_ROOT / "13_external_validation_dataset_selection_and_preparation",
])

# Notebook 14 v3.6 outputs were generated in one of these directories depending on the fast-resume path.
N14_CANDIDATES = [
    RESULTS_ROOT / "14_hygd_smdg_domain_shift_audit",
    RESULTS_ROOT / "14_hygd_smdg_domain_shift_audit_reviewer_ready",
]
N14_DIR = first_dir_with_file(
    N14_CANDIDATES,
    "tables/notebook14_v36_reviewer_question_handoff.csv",
) or first_existing_dir(N14_CANDIDATES)

# Notebook 15 v3.6 robustness outputs are appended to the expanded all-HYGD result directory.
N15_CANDIDATES = [
    RESULTS_ROOT / "15_aqpr_expanded_core_audit_all_hygd_v31",
    RESULTS_ROOT / "15_aqpr_faithfulness_sanity_and_perturbation_sensitivity_fixed_v2",
    RESULTS_ROOT / "15_aqpr_faithfulness_sanity_and_perturbation_sensitivity",
]
N15_DIR = first_dir_with_file(
    N15_CANDIDATES,
    "tables/notebook15_v36_outputs_for_notebook16.csv",
) or first_existing_dir(N15_CANDIDATES)

N13_TABLE_DIR = N13_DIR / "smdg_journal_ready_package" / "tables" if N13_DIR else None
N13_FIGURE_DIR = N13_DIR / "smdg_journal_ready_package" / "figures" if N13_DIR else None
N13_FIGURE_DATA_DIR = N13_DIR / "smdg_journal_ready_package" / "figure_data" if N13_DIR else None
N13_HANDOFF_JSON = N13_DIR / "smdg_journal_ready_package" / "notebook13_smdg_calibration_handoff_for_notebook16.json" if N13_DIR else None

N14_TABLE_DIR = N14_DIR / "tables" if N14_DIR else None
N14_FIGURE_DIRS = [N14_DIR / "figures", N14_DIR / "figure_data"] if N14_DIR else []
N14_HANDOFF_JSON = N14_DIR / "handoff" / "notebook14_quality_proxy_patient_aggregation_handoff_for_notebook16.json" if N14_DIR else None
N14_V36_HANDOFF_CSV = N14_TABLE_DIR / "notebook14_v36_reviewer_question_handoff.csv" if N14_TABLE_DIR else None
N14_V36_SNIPPETS_CSV = N14_TABLE_DIR / "notebook14_v36_manuscript_snippets_for_notebook16.csv" if N14_TABLE_DIR else None

N15_TABLE_DIR = N15_DIR / "tables" if N15_DIR else None
N15_FIGURE_DIR = N15_DIR / "figures" if N15_DIR else None
N15_HANDOFF_JSON = N15_DIR / "notebook15_expanded_claim_boundary_handoff.json" if N15_DIR else None
N15_V36_HANDOFF_JSON = N15_DIR / "notebook15_v36_robustness_handoff_for_notebook16.json" if N15_DIR else None
N15_V36_OUTPUTS_CSV = N15_TABLE_DIR / "notebook15_v36_outputs_for_notebook16.csv" if N15_TABLE_DIR else None

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 2",
    "status": "PASS" if (N13_DIR and N14_DIR and N15_DIR) else "PASS_WITH_MISSING_UPSTREAM_DIR",
    "notebook13_dir": str(N13_DIR) if N13_DIR else "",
    "notebook14_dir": str(N14_DIR) if N14_DIR else "",
    "notebook15_dir": str(N15_DIR) if N15_DIR else "",
    "notebook14_v36_handoff_exists": bool(exists_nonempty(N14_V36_HANDOFF_CSV)),
    "notebook15_v36_handoff_exists": bool(exists_nonempty(N15_V36_HANDOFF_JSON)),
}
with open(RESULTS_16_DIR / "notebook16_v36_cell02_upstream_discovery_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:
# Cell 3: v3.6 evidence source registry

registry_items = []

def add_source(role, priority, path, description):
    p = Path(path) if path else None
    df = safe_read_csv(p) if p and p.suffix.lower() == ".csv" else pd.DataFrame()
    payload = safe_read_json(p) if p and p.suffix.lower() == ".json" else {}
    registry_items.append({
        "role": role,
        "priority": priority,
        "description": description,
        "path": str(p) if p else "",
        "exists": bool(p.exists()) if p else False,
        "size_bytes": int(p.stat().st_size) if p and p.exists() else 0,
        "n_rows": int(len(df)) if not df.empty else 0,
        "n_columns": int(df.shape[1]) if not df.empty else 0,
        "json_keys": ";".join(list(payload.keys())[:40]) if payload else "",
        "columns": ";".join(map(str, df.columns[:60])) if not df.empty else "",
    })

# Notebook 13 — calibration.
if N13_TABLE_DIR:
    for role, priority, fname, desc in [
        ("n13_global_calibration", "main", "table_smdg_global_calibration_raw_vs_scaled_all_models.csv", "Global SMDG raw/scaled Brier/ECE by model."),
        ("n13_source_calibration_primary_compact", "main", "table_smdg_source_calibration_raw_vs_scaled_primary_model_compact.csv", "Primary-model per-source calibration compact table."),
        ("n13_source_calibration_primary_full", "supplement", "table_smdg_source_calibration_raw_vs_scaled_primary_model.csv", "Primary-model full per-source calibration table."),
        ("n13_source_calibration_all_models", "supplement", "table_smdg_source_calibration_raw_vs_scaled_all_models.csv", "All-model per-source calibration table."),
        ("n13_source_metric_eligibility", "supplement", "table_smdg_source_metric_eligibility_and_calibration_support.csv", "SMDG source metric eligibility and calibration support."),
    ]:
        add_source(role, priority, N13_TABLE_DIR / fname, desc)

if N13_FIGURE_DATA_DIR:
    for role, priority, fname, desc in [
        ("n13_reliability_bins", "supplement", "figure_data_smdg_reliability_diagram_bins_all_models.csv", "Reliability diagram bin data."),
        ("n13_source_calibration_delta_primary", "main", "figure_data_smdg_source_calibration_delta_primary_model.csv", "Primary-model source calibration deltas."),
        ("n13_global_calibration_delta", "main", "figure_data_smdg_global_calibration_delta_all_models.csv", "Global calibration deltas."),
    ]:
        add_source(role, priority, N13_FIGURE_DATA_DIR / fname, desc)

add_source("n13_calibration_handoff", "main", N13_HANDOFF_JSON, "Notebook 13 calibration handoff.")

# Notebook 14 — quality/proxy/aggregation/forensics.
if N14_TABLE_DIR:
    for role, priority, fname, desc in [
        ("n14_hygd_quality_summary", "main", "notebook14_hygd_quality_stratified_aqpr_summary.csv", "HYGD quality-stratified AQPR summary."),
        ("n14_quality_score_definition", "main", "notebook14_hygd_quality_score_definition.csv", "Quality score source and missingness."),
        ("n14_quality_thresholds", "main", "notebook14_hygd_quality_tertile_thresholds.csv", "Quality tertile thresholds and counts."),
        ("n14_proxy_quality_evidence", "main", "notebook14_hygd_proxy_quality_evidence_summary.csv", "Proxy strict-valid and distribution evidence."),
        ("n14_proxy_cdr_difference", "main", "notebook14_hygd_proxy_patient_cdr_gon_difference_bootstrap.csv", "Patient-level CDR proxy difference."),
        ("n14_proxy_failure_audit", "supplement", "notebook14_hygd_proxy_failure_audit.csv", "Proxy failure audit."),
        ("n14_patient_aggregation_compact", "main", "notebook14_hygd_patient_aggregation_robustness_compact.csv", "Patient aggregation robustness compact."),
        ("n14_patient_aggregation_long", "supplement", "notebook14_hygd_patient_aggregation_robustness_long.csv", "Patient aggregation robustness long."),
        ("n14_proxy_qc_by_quality_label", "supplement", "notebook14_hygd_proxy_qc_by_quality_and_label.csv", "Proxy QC by quality/label."),
        ("n14_endpoint_availability", "supplement", "notebook14_hygd_quality_endpoint_availability_audit.csv", "Endpoint availability."),
        ("n14_near_duplicate_threshold", "supplement", "notebook14_hygd_near_duplicate_threshold_summary.csv", "Near-duplicate threshold audit."),
        ("n14_exact_duplicate_groups", "supplement", "notebook14_hygd_exact_duplicate_groups.csv", "Exact duplicate groups."),
        ("n14_smdg_source_metrics", "main", "notebook14_smdg_source_prevalence_with_metrics.csv", "SMDG source-level transportability."),
        ("n14_smdg_label_policy", "supplement", "notebook14_smdg_label_harmonization_policy.csv", "SMDG label harmonization policy."),
        ("n14_reviewer_matrix", "main", "notebook14_reviewer_response_evidence_matrix.csv", "Notebook 14 reviewer response matrix."),
        ("n14_manuscript_summary", "main", "notebook14_manuscript_evidence_summary.csv", "Notebook 14 manuscript evidence summary."),
        ("n14_notebook16_actions", "main", "notebook14_to_notebook16_action_items.csv", "Notebook 16 action items."),
    ]:
        add_source(role, priority, N14_TABLE_DIR / fname, desc)

add_source("n14_quality_proxy_handoff", "main", N14_HANDOFF_JSON, "Notebook 14 quality/proxy/patient handoff.")

# Notebook 15 — expanded all-HYGD audit.
if N15_TABLE_DIR:
    for role, priority, fname, desc in [
        ("n15_all_hygd_cohort", "main", "notebook15_hygd_all_strict_valid_core_audit_cohort.csv", "All-HYGD strict-valid core audit cohort."),
        ("n15_model_load_audit", "supplement", "notebook15_model_load_audit.csv", "Model loading audit."),
        ("n15_saliency_degradation_ci", "main", "notebook15_expanded_stratified_saliency_degradation_ci.csv", "Saliency randomization degradation CIs."),
        ("n15_deletion_insertion_auc_ci", "main", "notebook15_expanded_stratified_deletion_insertion_auc_guided_minus_random_ci.csv", "Insertion/deletion AUC CIs."),
        ("n15_perturbation_odoc_random_ci", "main", "notebook15_expanded_stratified_perturbation_odoc_minus_random_ci.csv", "Perturbation OD/OC-minus-random CIs."),
        ("n15_perturbation_method_stability", "main", "notebook15_expanded_stratified_perturbation_method_stability.csv", "Perturbation method stability."),
        ("n15_core_endpoint_summary", "main", "notebook15_expanded_stratified_core_endpoint_summary.csv", "Expanded core endpoint summary."),
        ("n15_claim_boundary_full", "main", "notebook15_expanded_model_claim_boundary_full.csv", "Expanded model claim-boundary full table."),
        ("n15_claim_boundary_short", "main", "notebook15_expanded_model_claim_boundary_short.csv", "Expanded model claim-boundary short table."),
    ]:
        add_source(role, priority, N15_TABLE_DIR / fname, desc)

add_source("n15_expanded_claim_handoff", "main", N15_HANDOFF_JSON, "Notebook 15 expanded claim-boundary handoff.")

source_registry = pd.DataFrame(registry_items)
SOURCE_REGISTRY_CSV = AUDIT_DIR / "notebook16_v36_complete_evidence_source_registry.csv"
MISSING_SOURCES_CSV = AUDIT_DIR / "notebook16_v36_missing_critical_sources.csv"
source_registry.to_csv(SOURCE_REGISTRY_CSV, index=False)
missing = source_registry[source_registry["priority"].isin(["main", "supplement"]) & (~source_registry["exists"])].copy()
missing.to_csv(MISSING_SOURCES_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 3",
    "status": "PASS" if missing.empty else "PASS_WITH_MISSING_SOURCES",
    "source_registry_csv": str(SOURCE_REGISTRY_CSV),
    "missing_sources_csv": str(MISSING_SOURCES_CSV),
    "n_registered": int(len(source_registry)),
    "n_missing": int(len(missing)),
}
with open(RESULTS_16_DIR / "notebook16_v36_cell03_source_registry_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(source_registry[["role", "priority", "exists", "n_rows", "n_columns", "path"]].to_string(index=False))
print("\\nSummary:")
print(json.dumps(summary, indent=2))


# v3.6 reviewer-requested minor-revision extensions.
if N14_TABLE_DIR:
    for role, priority, fname, desc in [
        ("n14_v36_proxy_morphometric_baseline_vs_e2e", "main", "notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.csv", "Proxy CDR/morphometric baseline versus end-to-end HYGD classifiers."),
        ("n14_v36_proxy_morphometric_oof_predictions", "supplement", "notebook14_v36_proxy_morphometric_oof_predictions.csv", "OOF predictions for proxy morphometric baseline."),
        ("n14_v36_end_to_end_prediction_audit", "supplement", "notebook14_v36_hygd_end_to_end_prediction_audit.csv", "Raw/fallback end-to-end prediction audit for HYGD comparator table."),
        ("n14_v36_outside_od_features", "supplement", "notebook14_v36_outside_od_root_cause_features.csv", "Per-image outside-OD forensic features."),
        ("n14_v36_outside_od_univariate_auc", "main", "notebook14_v36_outside_od_root_cause_univariate_auc.csv", "Univariate outside-OD forensic feature AUROC."),
        ("n14_v36_outside_od_top_features", "main", "notebook14_v36_outside_od_root_cause_top_features.csv", "Top outside-OD forensic features."),
        ("n14_v36_border_corner_sanitized_manifest", "supplement", "notebook14_v36_border_corner_sanitized_input_manifest.csv", "Manifest for optional border/corner-sanitized inputs; no performance claims without re-inference."),
        ("n14_v36_reviewer_question_handoff", "main", "notebook14_v36_reviewer_question_handoff.csv", "Notebook 14 v3.6 reviewer-question handoff."),
        ("n14_v36_manuscript_snippets", "main", "notebook14_v36_manuscript_snippets_for_notebook16.csv", "Notebook 14 v3.6 manuscript snippets."),
    ]:
        add_source(role, priority, N14_TABLE_DIR / fname, desc)

if N15_TABLE_DIR:
    for role, priority, fname, desc in [
        ("n15_v36_endpoint_parameter_table", "main", "notebook15_aqpr_endpoint_parameter_table.csv", "AQPR endpoint parameter and reproducibility table."),
        ("n15_v36_auc_delta_formula", "main", "notebook15_insertion_deletion_auc_delta_formula_table.csv", "AUC-delta formula and curve-integration details."),
        ("n15_v36_full_curves_image", "supplement", "notebook15_full_insertion_deletion_curve_points_image_level.csv", "Full image-level insertion/deletion curve points."),
        ("n15_v36_full_curves_patient", "supplement", "notebook15_full_insertion_deletion_curve_points_patient_level.csv", "Full patient-level insertion/deletion curve points."),
        ("n15_v36_mr_saliency_summary", "main", "notebook15_matched_random_ablation_saliency_summary.csv", "Matched-random K/IoU saliency-enrichment ablation."),
        ("n15_v36_mr_perturbation_summary", "main", "notebook15_matched_random_ablation_perturbation_summary.csv", "Matched-random K/IoU perturbation-delta ablation."),
        ("n15_v36_secondary_occlusion_summary", "main", "notebook15_secondary_occlusion_attribution_summary.csv", "Secondary patch-occlusion attribution robustness check."),
        ("n15_v36_secondary_occlusion_curve_patient", "supplement", "notebook15_secondary_occlusion_deletion_insertion_curve_patient_level.csv", "Patch-occlusion patient-level insertion/deletion curves."),
        ("n15_v36_secondary_occlusion_auc_patient", "supplement", "notebook15_secondary_occlusion_deletion_insertion_auc_patient_level.csv", "Patch-occlusion patient-level AUC deltas."),
        ("n15_v36_outputs_for_notebook16", "main", "notebook15_v36_outputs_for_notebook16.csv", "Notebook 15 v3.6 output readiness inventory."),
        ("n15_v36_reviewer_response_matrix", "main", "notebook15_v36_reviewer_response_matrix.csv", "Notebook 15 v3.6 reviewer-response matrix."),
    ]:
        add_source(role, priority, N15_TABLE_DIR / fname, desc)

add_source("n15_v36_robustness_handoff", "main", N15_V36_HANDOFF_JSON, "Notebook 15 v3.6 robustness handoff for Notebook 16.")

source_registry = pd.DataFrame(registry_items)
SOURCE_REGISTRY_CSV = AUDIT_DIR / "notebook16_v36_complete_evidence_source_registry.csv"
MISSING_SOURCES_CSV = AUDIT_DIR / "notebook16_v36_missing_critical_sources.csv"
source_registry.to_csv(SOURCE_REGISTRY_CSV, index=False)
missing = source_registry[source_registry["priority"].isin(["main", "supplement"]) & (~source_registry["exists"])].copy()
missing.to_csv(MISSING_SOURCES_CSV, index=False)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 3",
    "status": "PASS" if missing.empty else "PASS_WITH_MISSING_SOURCES",
    "source_registry_csv": str(SOURCE_REGISTRY_CSV),
    "missing_sources_csv": str(MISSING_SOURCES_CSV),
    "n_registered": int(len(source_registry)),
    "n_missing": int(len(missing)),
    "v36_sources_registered": True,
}
with open(RESULTS_16_DIR / "notebook16_v36_cell03_source_registry_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nUpdated v3.6 source registry:")
print(source_registry[["role", "priority", "exists", "n_rows", "n_columns", "path"]].to_string(index=False))
print("\nSummary:")
print(json.dumps(summary, indent=2))


In [ ]:
# Cell 4: Main table 1 — Dataset roles and frozen component development

dataset_roles = pd.DataFrame([
    {
        "Block": "Classifier development",
        "Dataset/artifact": "AIROGS-light",
        "Evidence": "Internal balanced development subset; train/validation/locked-test split.",
        "Key numbers": "9540 images; train 8000, validation 770, locked test 770.",
        "AQPR role": "Train/select/freeze glaucoma classifier panel and fit temperature on validation only.",
        "Guardrail": "No HYGD/SMDG image used for training, threshold fitting, or source-specific recalibration.",
    },
    {
        "Block": "OD/OC proxy development",
        "Dataset/artifact": "REFUGE/PAPILA/RIGA",
        "Evidence": "External OD/OC segmentation sources; architecture screening and cross-dataset validation.",
        "Key numbers": "ResidualUNetLite; mean Dice 0.868; OD Dice 0.902; OC Dice 0.834; minimum-domain Dice 0.813.",
        "AQPR role": "Generate frozen OD/OC proxy masks for HYGD anatomical audit endpoints.",
        "Guardrail": "HYGD OD/OC masks are proxies, not manual HYGD ground truth.",
    },
    {
        "Block": "Classifier family screening",
        "Dataset/artifact": "AIROGS-light classifier panel",
        "Evidence": "Family-wise screening of representative CNN/modern ConvNet/dense/efficient/transformer models.",
        "Key numbers": "EfficientNetB0 primary: validation AUROC 0.985; locked-test AUROC 0.979 [0.970, 0.987]; balanced accuracy 0.930 [0.911, 0.948].",
        "AQPR role": "Primary model plus architecture-contrast panel.",
        "Guardrail": "Frozen external audit evaluates models rather than reselecting them on HYGD/SMDG.",
    },
    {
        "Block": "External AQPR audit",
        "Dataset/artifact": "HYGD",
        "Evidence": "Patient-aware external audit with quality scores and OD/OC proxy-QC.",
        "Key numbers": "747 images; 288 patients; 548 GON+ and 199 GON− images.",
        "AQPR role": "Anatomy, quality, patient aggregation, saliency/deletion/perturbation, and claim-boundary audit.",
        "Guardrail": "External audit only; no training or calibration fitting.",
    },
    {
        "Block": "Source transportability audit",
        "Dataset/artifact": "SMDG",
        "Evidence": "Source-level external validation, label harmonization, and per-source calibration diagnostics.",
        "Key numbers": "8627 images; 18 source entries.",
        "AQPR role": "Source-wise AUROC, Brier/ECE raw vs scaled, and reliability diagrams.",
        "Guardrail": "Transportability audit, not homogeneous clinical validation.",
    },
])

DATASET_ROLES_CSV = TABLE_DIR / "final_main_table_dataset_roles_and_frozen_component_development.csv"
DATASET_ROLES_TEX = LATEX_DIR / "final_main_table_dataset_roles_and_frozen_component_development.tex"
save_csv_md(dataset_roles, DATASET_ROLES_CSV)
write_latex_table(dataset_roles, DATASET_ROLES_TEX, "Dataset roles, frozen component development, and audit guardrails.", "tab:dataset_roles_component_development")
print(dataset_roles.to_string(index=False))


In [ ]:
# Cell 5: Main table 2 — HYGD quality, proxy-QC, and patient aggregation evidence

quality = safe_read_csv(N14_TABLE_DIR / "notebook14_hygd_quality_stratified_aqpr_summary.csv" if N14_TABLE_DIR else None)
thresholds = safe_read_csv(N14_TABLE_DIR / "notebook14_hygd_quality_tertile_thresholds.csv" if N14_TABLE_DIR else None)
cdr_diff = safe_read_csv(N14_TABLE_DIR / "notebook14_hygd_proxy_patient_cdr_gon_difference_bootstrap.csv" if N14_TABLE_DIR else None)
aggregation = safe_read_csv(N14_TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_compact.csv" if N14_TABLE_DIR else None)

if quality.empty:
    final_quality = pd.DataFrame()
else:
    q = quality.copy()
    if not thresholds.empty and "quality_tertile" in thresholds.columns and "quality_tertile" in q.columns:
        merge_cols = select_cols(thresholds, ["quality_tertile", "score_min", "score_median", "score_max", "strict_proxy_valid_fraction"])
        if merge_cols:
            q = q.merge(thresholds[merge_cols], on="quality_tertile", how="left", suffixes=("", "_threshold"))

    preferred = [
        "quality_tertile", "n_images", "n_patients", "n_gon_positive", "n_gon_negative",
        "score_min", "score_median", "score_max",
        "strict_proxy_fraction", "strict_proxy_valid_fraction",
        "cdr_proxy_mean", "primary_auroc", "primary_patient_auroc",
        "primary_ece_10bin", "primary_patient_ece_10bin",
        "primary_ig_od_oc_observed_minus_random_mean",
        "primary_od_oc_blur_mean_probability_drop_vs_original",
        "primary_od_oc_minus_random_blur_mean_probability_drop",
    ]
    final_quality = q[select_cols(q, preferred)].rename(columns={
        "quality_tertile": "Quality",
        "n_images": "Images",
        "n_patients": "Patients",
        "n_gon_positive": "GON+",
        "n_gon_negative": "GON−",
        "score_min": "Q score min",
        "score_median": "Q score median",
        "score_max": "Q score max",
        "strict_proxy_fraction": "Strict proxy fraction",
        "strict_proxy_valid_fraction": "Strict proxy fraction",
        "cdr_proxy_mean": "Mean CDR proxy",
        "primary_auroc": "Image AUROC",
        "primary_patient_auroc": "Patient AUROC",
        "primary_ece_10bin": "Image ECE",
        "primary_patient_ece_10bin": "Patient ECE",
        "primary_ig_od_oc_observed_minus_random_mean": "IG OD+OC enrichment",
        "primary_od_oc_blur_mean_probability_drop_vs_original": "OD+OC blur drop",
        "primary_od_oc_minus_random_blur_mean_probability_drop": "OD+OC minus random blur",
    })

if cdr_diff.empty:
    proxy_cdr_main = pd.DataFrame()
else:
    proxy_cdr_main = cdr_diff.rename(columns={
        "positive_mean": "GON+ mean CDR proxy",
        "negative_mean": "GON− mean CDR proxy",
        "mean_difference_positive_minus_negative": "Mean difference",
        "ci95_low": "CI low",
        "ci95_high": "CI high",
        "n_positive_units": "GON+ patients",
        "n_negative_units": "GON− patients",
    })
    keep = select_cols(proxy_cdr_main, [
        "proxy_metric", "GON+ patients", "GON− patients", "GON+ mean CDR proxy",
        "GON− mean CDR proxy", "Mean difference", "CI low", "CI high", "claim_boundary"
    ])
    proxy_cdr_main = proxy_cdr_main[keep]

FINAL_QUALITY_CSV = TABLE_DIR / "final_main_table_hygd_quality_proxy_patient_aqpr.csv"
FINAL_QUALITY_TEX = LATEX_DIR / "final_main_table_hygd_quality_proxy_patient_aqpr.tex"
PROXY_CDR_MAIN_CSV = TABLE_DIR / "final_main_table_hygd_proxy_cdr_face_validity.csv"
PROXY_CDR_MAIN_TEX = LATEX_DIR / "final_main_table_hygd_proxy_cdr_face_validity.tex"
FINAL_AGGREGATION_CSV = TABLE_DIR / "final_supp_table_hygd_patient_aggregation_robustness.csv"
FINAL_AGGREGATION_TEX = LATEX_DIR / "final_supp_table_hygd_patient_aggregation_robustness.tex"

save_csv_md(final_quality, FINAL_QUALITY_CSV)
save_csv_md(proxy_cdr_main, PROXY_CDR_MAIN_CSV)
save_csv_md(aggregation, FINAL_AGGREGATION_CSV)
write_latex_table(final_quality, FINAL_QUALITY_TEX, "HYGD quality-stratified AQPR and OD/OC proxy evidence.", "tab:hygd_quality_aqpr")
write_latex_table(proxy_cdr_main, PROXY_CDR_MAIN_TEX, "HYGD patient-level CDR proxy face-validity evidence.", "tab:hygd_proxy_cdr")
write_latex_table(aggregation, FINAL_AGGREGATION_TEX, "Patient-level aggregation robustness on HYGD.", "tab:patient_aggregation_robustness")

print("HYGD quality/proxy main table:")
print(final_quality.to_string(index=False) if not final_quality.empty else "Unavailable")
print("\\nHYGD proxy CDR face-validity table:")
print(proxy_cdr_main.to_string(index=False) if not proxy_cdr_main.empty else "Unavailable")
print("\\nPatient aggregation robustness:")
print(aggregation.to_string(index=False) if not aggregation.empty else "Unavailable")


In [ ]:
# Cell 6: Main table 3 — Expanded all-HYGD model claim-boundary audit

claim_path = first_existing_file([
    N15_TABLE_DIR / "notebook15_expanded_model_claim_boundary_full.csv" if N15_TABLE_DIR else None,
    N15_TABLE_DIR / "notebook15_model_claim_boundary_full.csv" if N15_TABLE_DIR else None,
    N15_TABLE_DIR / "notebook15_model_claim_boundary_audit.csv" if N15_TABLE_DIR else None,
])
claim = safe_read_csv(claim_path)

if claim.empty:
    final_claim_main = pd.DataFrame()
    final_claim_numeric = pd.DataFrame()
else:
    c = claim.copy()
    final_claim_main = c.rename(columns={
        "model_name": "Model",
        "saliency_sanity_status": "Saliency sanity",
        "deletion_insertion_faithfulness_status": "Deletion/insertion faithfulness",
        "deletion_faithfulness_status": "Deletion/insertion faithfulness",
        "perturbation_stability_status": "Perturbation stability",
        "overall_aqpr_convergence_status": "Overall AQPR convergence",
        "manuscript_claim_boundary": "Manuscript claim boundary",
        "safe_manuscript_claim": "Manuscript claim boundary",
    })
    final_claim_main = final_claim_main[select_cols(final_claim_main, [
        "Model", "Saliency sanity", "Deletion/insertion faithfulness", "Perturbation stability",
        "Overall AQPR convergence", "Manuscript claim boundary"
    ])]

    numeric_cols = select_cols(c, [
        "model_name",
        "saliency_head_delta_mean", "saliency_head_delta_ci95_low", "saliency_head_delta_ci95_high",
        "saliency_last_block_delta_mean", "saliency_last_block_delta_ci95_low", "saliency_last_block_delta_ci95_high",
        "deletion_auc_delta_mean", "deletion_auc_delta_ci95_low", "deletion_auc_delta_ci95_high",
        "insertion_auc_delta_mean", "insertion_auc_delta_ci95_low", "insertion_auc_delta_ci95_high",
        "blur_supported_settings", "blur_total_settings",
        "mean_fill_supported_settings", "mean_fill_total_settings",
        "gaussian_noise_supported_settings", "gaussian_noise_total_settings",
    ])
    final_claim_numeric = c[numeric_cols].rename(columns={"model_name": "Model"}) if numeric_cols else pd.DataFrame()

FINAL_CLAIM_MAIN_CSV = TABLE_DIR / "final_main_table_expanded_model_claim_boundary.csv"
FINAL_CLAIM_MAIN_TEX = LATEX_DIR / "final_main_table_expanded_model_claim_boundary.tex"
FINAL_CLAIM_NUMERIC_CSV = TABLE_DIR / "final_supp_table_expanded_model_claim_boundary_numeric_anchors.csv"
FINAL_CLAIM_NUMERIC_TEX = LATEX_DIR / "final_supp_table_expanded_model_claim_boundary_numeric_anchors.tex"

save_csv_md(final_claim_main, FINAL_CLAIM_MAIN_CSV)
save_csv_md(final_claim_numeric, FINAL_CLAIM_NUMERIC_CSV)
write_latex_table(final_claim_main, FINAL_CLAIM_MAIN_TEX, "Expanded all-HYGD AQPR claim-boundary audit.", "tab:expanded_claim_boundary")
write_latex_table(final_claim_numeric, FINAL_CLAIM_NUMERIC_TEX, "Numerical anchors for expanded all-HYGD AQPR claim boundaries.", "tab:expanded_claim_numeric")

print("Expanded claim-boundary main table:")
print(final_claim_main.to_string(index=False) if not final_claim_main.empty else "Unavailable")
print("\\nNumerical anchors:")
print(final_claim_numeric.to_string(index=False) if not final_claim_numeric.empty else "Unavailable")


In [ ]:
# Cell 7: Main table 4 — SMDG source-level transportability and per-source calibration

smdg = safe_read_csv(N14_TABLE_DIR / "notebook14_smdg_source_prevalence_with_metrics.csv" if N14_TABLE_DIR else None)
if smdg.empty:
    final_smdg_main = pd.DataFrame()
else:
    s = smdg.copy()
    if {"auroc", "auroc_ci95_low", "auroc_ci95_high"}.issubset(s.columns):
        s["AUROC [95% CI]"] = s.apply(lambda r: fmt_ci(r["auroc"], r["auroc_ci95_low"], r["auroc_ci95_high"], 3), axis=1)
    if {"balanced_accuracy", "balanced_accuracy_ci95_low", "balanced_accuracy_ci95_high"}.issubset(s.columns):
        s["Balanced accuracy [95% CI]"] = s.apply(lambda r: fmt_ci(r["balanced_accuracy"], r["balanced_accuracy_ci95_low"], r["balanced_accuracy_ci95_high"], 3), axis=1)
    source_col = "external_source_dataset" if "external_source_dataset" in s.columns else "source"
    final_smdg_main = s.rename(columns={
        source_col: "Source",
        "n_images": "Images",
        "n_positive": "Positive",
        "n_negative": "Negative",
        "positive_prevalence": "Positive prevalence",
        "brier": "Brier",
        "ece_10bin": "ECE",
        "below_chance_auroc_flag": "Below-chance AUROC flag",
    })
    final_smdg_main = final_smdg_main[select_cols(final_smdg_main, [
        "Source", "Images", "Positive", "Negative", "Positive prevalence",
        "AUROC [95% CI]", "Balanced accuracy [95% CI]", "Brier", "ECE", "Below-chance AUROC flag"
    ])]

global_cal = safe_read_csv(N13_TABLE_DIR / "table_smdg_global_calibration_raw_vs_scaled_all_models.csv" if N13_TABLE_DIR else None)
source_cal = safe_read_csv(N13_TABLE_DIR / "table_smdg_source_calibration_raw_vs_scaled_primary_model_compact.csv" if N13_TABLE_DIR else None)
eligibility = safe_read_csv(N13_TABLE_DIR / "table_smdg_source_metric_eligibility_and_calibration_support.csv" if N13_TABLE_DIR else None)

if not source_cal.empty:
    compact_cols = select_cols(source_cal, [
        "Source", "Images", "Prev.", "Eligible", "Raw AUROC",
        "Raw Brier", "Scaled Brier", "Delta Brier",
        "Raw ECE-15", "Scaled ECE-15", "Delta ECE-15", "Calibration effect"
    ])
    final_source_cal = source_cal[compact_cols] if compact_cols else source_cal.copy()
else:
    final_source_cal = pd.DataFrame()

FINAL_SMDG_TRANSPORT_CSV = TABLE_DIR / "final_main_table_smdg_source_transportability.csv"
FINAL_SMDG_TRANSPORT_TEX = LATEX_DIR / "final_main_table_smdg_source_transportability.tex"
FINAL_SMDG_CAL_SOURCE_CSV = TABLE_DIR / "final_main_table_smdg_source_calibration_primary.csv"
FINAL_SMDG_CAL_SOURCE_TEX = LATEX_DIR / "final_main_table_smdg_source_calibration_primary.tex"
FINAL_SMDG_CAL_GLOBAL_CSV = TABLE_DIR / "final_supp_table_smdg_global_calibration.csv"
FINAL_SMDG_CAL_GLOBAL_TEX = LATEX_DIR / "final_supp_table_smdg_global_calibration.tex"
FINAL_SMDG_ELIGIBILITY_CSV = TABLE_DIR / "final_supp_table_smdg_source_metric_eligibility.csv"

save_csv_md(final_smdg_main, FINAL_SMDG_TRANSPORT_CSV)
save_csv_md(final_source_cal, FINAL_SMDG_CAL_SOURCE_CSV)
save_csv_md(global_cal, FINAL_SMDG_CAL_GLOBAL_CSV)
save_csv_md(eligibility, FINAL_SMDG_ELIGIBILITY_CSV)
write_latex_table(final_smdg_main, FINAL_SMDG_TRANSPORT_TEX, "SMDG source-level transportability.", "tab:smdg_transportability")
write_latex_table(final_source_cal, FINAL_SMDG_CAL_SOURCE_TEX, "SMDG source-level calibration for the primary model.", "tab:smdg_source_calibration")
write_latex_table(global_cal, FINAL_SMDG_CAL_GLOBAL_TEX, "Global SMDG raw vs temperature-scaled calibration.", "tab:smdg_global_calibration")

print("SMDG source-level transportability:")
print(final_smdg_main.to_string(index=False) if not final_smdg_main.empty else "Unavailable")
print("\\nSMDG source-level calibration:")
print(final_source_cal.to_string(index=False) if not final_source_cal.empty else "Unavailable")
print("\\nSMDG global calibration:")
print(global_cal.to_string(index=False) if not global_cal.empty else "Unavailable")


In [ ]:
# Cell 8: Supplementary evidence package

supp_sources = [
    ("S01_smdg_global_calibration_raw_vs_scaled", N13_TABLE_DIR / "table_smdg_global_calibration_raw_vs_scaled_all_models.csv" if N13_TABLE_DIR else None),
    ("S02_smdg_source_calibration_primary_full", N13_TABLE_DIR / "table_smdg_source_calibration_raw_vs_scaled_primary_model.csv" if N13_TABLE_DIR else None),
    ("S03_smdg_source_calibration_all_models", N13_TABLE_DIR / "table_smdg_source_calibration_raw_vs_scaled_all_models.csv" if N13_TABLE_DIR else None),
    ("S04_smdg_source_metric_eligibility", N13_TABLE_DIR / "table_smdg_source_metric_eligibility_and_calibration_support.csv" if N13_TABLE_DIR else None),
    ("S05_smdg_reliability_bins", N13_FIGURE_DATA_DIR / "figure_data_smdg_reliability_diagram_bins_all_models.csv" if N13_FIGURE_DATA_DIR else None),
    ("S06_hygd_quality_score_definition", N14_TABLE_DIR / "notebook14_hygd_quality_score_definition.csv" if N14_TABLE_DIR else None),
    ("S07_hygd_quality_tertile_thresholds", N14_TABLE_DIR / "notebook14_hygd_quality_tertile_thresholds.csv" if N14_TABLE_DIR else None),
    ("S08_hygd_proxy_quality_evidence", N14_TABLE_DIR / "notebook14_hygd_proxy_quality_evidence_summary.csv" if N14_TABLE_DIR else None),
    ("S09_hygd_proxy_cdr_difference", N14_TABLE_DIR / "notebook14_hygd_proxy_patient_cdr_gon_difference_bootstrap.csv" if N14_TABLE_DIR else None),
    ("S10_hygd_proxy_failure_audit", N14_TABLE_DIR / "notebook14_hygd_proxy_failure_audit.csv" if N14_TABLE_DIR else None),
    ("S11_hygd_patient_aggregation_robustness_long", N14_TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_long.csv" if N14_TABLE_DIR else None),
    ("S12_hygd_patient_aggregation_robustness_compact", N14_TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_compact.csv" if N14_TABLE_DIR else None),
    ("S13_hygd_near_duplicate_threshold_summary", N14_TABLE_DIR / "notebook14_hygd_near_duplicate_threshold_summary.csv" if N14_TABLE_DIR else None),
    ("S14_hygd_exact_duplicate_groups", N14_TABLE_DIR / "notebook14_hygd_exact_duplicate_groups.csv" if N14_TABLE_DIR else None),
    ("S15_smdg_label_harmonization_policy", N14_TABLE_DIR / "notebook14_smdg_label_harmonization_policy.csv" if N14_TABLE_DIR else None),
    ("S16_notebook15_all_hygd_audit_cohort", N15_TABLE_DIR / "notebook15_hygd_all_strict_valid_core_audit_cohort.csv" if N15_TABLE_DIR else None),
    ("S17_saliency_sanity_degradation_ci", N15_TABLE_DIR / "notebook15_expanded_stratified_saliency_degradation_ci.csv" if N15_TABLE_DIR else None),
    ("S18_deletion_insertion_auc_guided_minus_random_ci", N15_TABLE_DIR / "notebook15_expanded_stratified_deletion_insertion_auc_guided_minus_random_ci.csv" if N15_TABLE_DIR else None),
    ("S19_perturbation_odoc_minus_random_ci", N15_TABLE_DIR / "notebook15_expanded_stratified_perturbation_odoc_minus_random_ci.csv" if N15_TABLE_DIR else None),
    ("S20_perturbation_method_stability", N15_TABLE_DIR / "notebook15_expanded_stratified_perturbation_method_stability.csv" if N15_TABLE_DIR else None),
    ("S21_expanded_claim_boundary_full", N15_TABLE_DIR / "notebook15_expanded_model_claim_boundary_full.csv" if N15_TABLE_DIR else None),
]
rows = []
for sid, src in supp_sources:
    copied = copy_if_exists(src, SUPP_TABLE_DIR, new_name=f"{sid}.csv") if src else ""
    df = safe_read_csv(copied) if copied else pd.DataFrame()
    rows.append({
        "supplementary_id": sid,
        "source_path": str(src) if src else "",
        "copied_path": copied,
        "available": bool(copied),
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]) if not df.empty else 0,
    })
supp_inventory = pd.DataFrame(rows)
SUPP_INVENTORY_CSV = SUPP_TABLE_DIR / "supplementary_table_inventory.csv"
supp_inventory.to_csv(SUPP_INVENTORY_CSV, index=False)
print(supp_inventory.to_string(index=False))


# v3.6 reviewer-requested supplementary evidence.
supp_sources_v36 = [
    ("S22_v36_endpoint_parameter_table", N15_TABLE_DIR / "notebook15_aqpr_endpoint_parameter_table.csv" if N15_TABLE_DIR else None),
    ("S23_v36_auc_delta_formula_table", N15_TABLE_DIR / "notebook15_insertion_deletion_auc_delta_formula_table.csv" if N15_TABLE_DIR else None),
    ("S24_v36_full_insertion_deletion_curves_patient", N15_TABLE_DIR / "notebook15_full_insertion_deletion_curve_points_patient_level.csv" if N15_TABLE_DIR else None),
    ("S25_v36_matched_random_saliency_ablation", N15_TABLE_DIR / "notebook15_matched_random_ablation_saliency_summary.csv" if N15_TABLE_DIR else None),
    ("S26_v36_matched_random_perturbation_ablation", N15_TABLE_DIR / "notebook15_matched_random_ablation_perturbation_summary.csv" if N15_TABLE_DIR else None),
    ("S27_v36_secondary_patch_occlusion_summary", N15_TABLE_DIR / "notebook15_secondary_occlusion_attribution_summary.csv" if N15_TABLE_DIR else None),
    ("S28_v36_secondary_patch_occlusion_auc_patient", N15_TABLE_DIR / "notebook15_secondary_occlusion_deletion_insertion_auc_patient_level.csv" if N15_TABLE_DIR else None),
    ("S29_v36_proxy_morphometric_vs_end_to_end_hygd", N14_TABLE_DIR / "notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.csv" if N14_TABLE_DIR else None),
    ("S30_v36_outside_od_root_cause_top_features", N14_TABLE_DIR / "notebook14_v36_outside_od_root_cause_top_features.csv" if N14_TABLE_DIR else None),
    ("S31_v36_outside_od_root_cause_univariate_auc", N14_TABLE_DIR / "notebook14_v36_outside_od_root_cause_univariate_auc.csv" if N14_TABLE_DIR else None),
    ("S32_v36_outside_od_root_cause_features", N14_TABLE_DIR / "notebook14_v36_outside_od_root_cause_features.csv" if N14_TABLE_DIR else None),
    ("S33_v36_border_corner_sanitized_manifest", N14_TABLE_DIR / "notebook14_v36_border_corner_sanitized_input_manifest.csv" if N14_TABLE_DIR else None),
]

v36_rows = []
for sid, src in supp_sources_v36:
    copied = copy_if_exists(src, SUPP_TABLE_DIR, new_name=f"{sid}.csv") if src else ""
    df = safe_read_csv(copied) if copied else pd.DataFrame()
    v36_rows.append({
        "supplementary_id": sid,
        "source_path": str(src) if src else "",
        "copied_path": copied,
        "available": bool(copied),
        "n_rows": int(len(df)),
        "n_columns": int(df.shape[1]) if not df.empty else 0,
    })

v36_inventory = pd.DataFrame(v36_rows)
supp_inventory = pd.concat([supp_inventory, v36_inventory], ignore_index=True)
supp_inventory.to_csv(SUPP_INVENTORY_CSV, index=False)
print("\nAdded v3.6 supplementary evidence:")
print(v36_inventory.to_string(index=False))


In [ ]:
# Cell 9: v3.6 reviewer-driven compact tables and manuscript snippets

# Inputs from Notebook 15 v3.6.
endpoint_params = safe_read_csv(N15_TABLE_DIR / "notebook15_aqpr_endpoint_parameter_table.csv" if N15_TABLE_DIR else None)
auc_formula = safe_read_csv(N15_TABLE_DIR / "notebook15_insertion_deletion_auc_delta_formula_table.csv" if N15_TABLE_DIR else None)
mr_saliency = safe_read_csv(N15_TABLE_DIR / "notebook15_matched_random_ablation_saliency_summary.csv" if N15_TABLE_DIR else None)
mr_perturb = safe_read_csv(N15_TABLE_DIR / "notebook15_matched_random_ablation_perturbation_summary.csv" if N15_TABLE_DIR else None)
occ_summary = safe_read_csv(N15_TABLE_DIR / "notebook15_secondary_occlusion_attribution_summary.csv" if N15_TABLE_DIR else None)

# Inputs from Notebook 14 v3.6.
cdr_baseline = safe_read_csv(N14_TABLE_DIR / "notebook14_v36_proxy_morphometric_baseline_vs_end_to_end_hygd.csv" if N14_TABLE_DIR else None)
outside_top = safe_read_csv(N14_TABLE_DIR / "notebook14_v36_outside_od_root_cause_top_features.csv" if N14_TABLE_DIR else None)
outside_auc = safe_read_csv(N14_TABLE_DIR / "notebook14_v36_outside_od_root_cause_univariate_auc.csv" if N14_TABLE_DIR else None)
n14_snippets = safe_read_csv(N14_V36_SNIPPETS_CSV)

# Compact matched-random stability table.
mr_rows = []
if not mr_saliency.empty:
    for model, g in mr_saliency.groupby("model_name"):
        val_col = "observed_minus_random_mean"
        if val_col not in g.columns:
            # fallback to first numeric column containing observed/random or enrichment.
            candidates = [c for c in g.columns if any(s in c.lower() for s in ["minus", "enrichment", "delta"]) and pd.api.types.is_numeric_dtype(g[c])]
            val_col = candidates[0] if candidates else None
        if val_col:
            mr_rows.append({
                "analysis": "saliency_enrichment",
                "model": model,
                "n_settings": len(g),
                "mean_effect_min": float(g[val_col].min()),
                "mean_effect_max": float(g[val_col].max()),
                "direction_stable": bool((g[val_col] > 0).all()),
                "settings": "K=32/64/128; max IoU=0.01/0.03/0.05; FOV containment fixed",
                "interpretation": "OD/OC enrichment direction preserved across matched-random settings.",
            })
if not mr_perturb.empty:
    for model, g in mr_perturb.groupby("model_name"):
        candidates = [c for c in g.columns if any(s in c.lower() for s in ["minus_random", "delta", "drop"]) and pd.api.types.is_numeric_dtype(g[c])]
        val_col = candidates[0] if candidates else None
        if val_col:
            mr_rows.append({
                "analysis": "perturbation_delta",
                "model": model,
                "n_settings": len(g),
                "mean_effect_min": float(g[val_col].min()),
                "mean_effect_max": float(g[val_col].max()),
                "direction_stable": bool((g[val_col] >= 0).all() or (g[val_col] <= 0).all()),
                "settings": "Sensitivity subset; K=8/16/32; max IoU=0.01/0.03/0.05; blur setting fixed",
                "interpretation": "Small sensitivity check for OD/OC-minus-random perturbation delta; main endpoint remains all-HYGD fixed setting.",
            })
mr_compact = pd.DataFrame(mr_rows)

# Compact secondary attribution table.
occ_compact = occ_summary.copy()
if not occ_compact.empty:
    keep = select_cols(occ_compact, [
        "model_name", "xai_method", "summary_name", "curve_type",
        "mean", "ci95_low", "ci95_high", "n_units", "n_images", "n_patients",
        "interpretation"
    ])
    occ_compact = occ_compact[keep] if keep else occ_compact

# CDR baseline compact table.
cdr_compact = cdr_baseline.copy()
if not cdr_compact.empty:
    keep = select_cols(cdr_compact, [
        "comparison_family", "model_or_feature", "level", "score_definition",
        "n_units", "n_positive", "n_negative", "auroc", "auprc", "brier",
        "selection_or_training", "manuscript_interpretation"
    ])
    cdr_compact = cdr_compact[keep] if keep else cdr_compact

# Outside-OD top features compact table.
outside_compact = outside_top.copy()
if not outside_compact.empty:
    keep = select_cols(outside_compact, [
        "feature", "level", "n_units", "n_positive", "n_negative",
        "auroc", "informative_auc", "higher_feature_associated_with", "interpretation"
    ])
    outside_compact = outside_compact[keep] if keep else outside_compact

V36_MR_COMPACT_CSV = TABLE_DIR / "final_v36_table_matched_random_robustness_compact.csv"
V36_MR_COMPACT_TEX = LATEX_DIR / "final_v36_table_matched_random_robustness_compact.tex"
V36_OCC_COMPACT_CSV = TABLE_DIR / "final_v36_table_secondary_patch_occlusion_robustness.csv"
V36_OCC_COMPACT_TEX = LATEX_DIR / "final_v36_table_secondary_patch_occlusion_robustness.tex"
V36_CDR_COMPACT_CSV = TABLE_DIR / "final_v36_table_proxy_morphometric_vs_end_to_end.csv"
V36_CDR_COMPACT_TEX = LATEX_DIR / "final_v36_table_proxy_morphometric_vs_end_to_end.tex"
V36_OUTSIDE_COMPACT_CSV = TABLE_DIR / "final_v36_table_outside_od_root_cause_top_features.csv"
V36_OUTSIDE_COMPACT_TEX = LATEX_DIR / "final_v36_table_outside_od_root_cause_top_features.tex"
V36_ENDPOINT_PARAMS_CSV = TABLE_DIR / "final_v36_table_endpoint_parameters.csv"
V36_AUC_FORMULA_CSV = TABLE_DIR / "final_v36_table_auc_delta_formula.csv"

save_csv_md(endpoint_params, V36_ENDPOINT_PARAMS_CSV)
save_csv_md(auc_formula, V36_AUC_FORMULA_CSV)
save_csv_md(mr_compact, V36_MR_COMPACT_CSV)
save_csv_md(occ_compact, V36_OCC_COMPACT_CSV)
save_csv_md(cdr_compact, V36_CDR_COMPACT_CSV)
save_csv_md(outside_compact, V36_OUTSIDE_COMPACT_CSV)

write_latex_table(mr_compact, V36_MR_COMPACT_TEX, "Matched-random robustness checks for AQPR endpoints.", "tab:v36_matched_random_robustness")
write_latex_table(occ_compact, V36_OCC_COMPACT_TEX, "Secondary patch-occlusion attribution robustness check.", "tab:v36_secondary_patch_occlusion")
write_latex_table(cdr_compact, V36_CDR_COMPACT_TEX, "Proxy morphometric baselines versus frozen end-to-end classifiers on HYGD.", "tab:v36_proxy_morphometric_baseline")
write_latex_table(outside_compact, V36_OUTSIDE_COMPACT_TEX, "Outside-OD forensic feature associations.", "tab:v36_outside_od_forensics")

# Manuscript-ready v3.6 fragments.
def _best_num(df, col, default="NA", decimals=3):
    try:
        if df.empty or col not in df.columns:
            return default
        val = pd.to_numeric(df[col], errors="coerce").dropna().iloc[0]
        return f"{float(val):.{decimals}f}"
    except Exception:
        return default

snippets = [
    {
        "fragment_name": "v36_endpoint_definition_fragment",
        "text": "Insertion/deletion AUC-delta was computed as the trapezoidal area under the saliency-guided probability curve minus the matched-random curve over fixed fractions 0.00, 0.05, 0.10, 0.20, 0.30, 0.40, and 0.50. Positive heatmap energy was defined as the normalized sum of positive attribution inside the OD/OC proxy region.",
        "use_location": "Methods or Supplement",
    },
    {
        "fragment_name": "v36_matched_random_robustness_fragment",
        "text": "Matched-random sensitivity checks varying K and maximum OD/OC overlap preserved the direction of OD/OC saliency enrichment; perturbation-delta sensitivity was evaluated on a balanced subset and treated as a robustness check rather than a primary endpoint.",
        "use_location": "Results/Supplement",
    },
    {
        "fragment_name": "v36_secondary_attribution_fragment",
        "text": "A secondary patch-occlusion attribution check on a balanced 180-image HYGD subset showed positive OD/OC enrichment and non-zero insertion/deletion deltas, supporting that the main IG-based audit was not solely an implementation artifact. This subset analysis was not used as the primary AQPR endpoint.",
        "use_location": "Results/Supplement",
    },
    {
        "fragment_name": "v36_cdr_baseline_fragment",
        "text": "Proxy CDR and shallow proxy-morphometric baselines provided meaningful segmentation-first anatomical baselines on HYGD, but they did not replace the end-to-end audited classifiers or the broader AQPR evidence streams covering localization, functional dependence, calibration, quality, patient aggregation, and transportability.",
        "use_location": "Results/Discussion",
    },
    {
        "fragment_name": "v36_outside_od_forensics_fragment",
        "text": "Outside-OD forensic features, including corner-edge content, vignetting-related contrast, FOV area, saturation, and peripheral intensity statistics, were associated with HYGD labels. These associations support interpretation of outside-OD performance as a dataset-artifact/shortcut warning, without proving causal reliance on any single cue.",
        "use_location": "Results/Limitations",
    },
]
snippets_df = pd.DataFrame(snippets)
V36_SNIPPETS_CSV = TEXT_DIR / "manuscript_ready_fragments_v36_minor_revision.csv"
V36_SNIPPETS_MD = TEXT_DIR / "manuscript_ready_fragments_v36_minor_revision.md"
save_csv_md(snippets_df, V36_SNIPPETS_CSV, V36_SNIPPETS_MD)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 9",
    "status": "PASS",
    "compact_tables": {
        "matched_random": str(V36_MR_COMPACT_CSV),
        "secondary_patch_occlusion": str(V36_OCC_COMPACT_CSV),
        "proxy_morphometric_baseline": str(V36_CDR_COMPACT_CSV),
        "outside_od_forensics": str(V36_OUTSIDE_COMPACT_CSV),
        "endpoint_parameters": str(V36_ENDPOINT_PARAMS_CSV),
        "auc_delta_formula": str(V36_AUC_FORMULA_CSV),
    },
    "manuscript_snippets": str(V36_SNIPPETS_CSV),
}
with open(RESULTS_16_DIR / "notebook16_v36_cell09_minor_revision_tables_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("v3.6 compact reviewer-driven tables generated:")
print(json.dumps(summary, indent=2))
print("\nMatched-random compact:")
print(mr_compact.to_string(index=False) if not mr_compact.empty else "Unavailable")
print("\nSecondary occlusion compact:")
print(occ_compact.to_string(index=False) if not occ_compact.empty else "Unavailable")
print("\nProxy morphometric baseline compact:")
print(cdr_compact.to_string(index=False) if not cdr_compact.empty else "Unavailable")
print("\nOutside-OD top features:")
print(outside_compact.to_string(index=False) if not outside_compact.empty else "Unavailable")


In [ ]:
# Cell 10: Figure inventory and copied figure candidates

figure_candidates = []

if N13_FIGURE_DIR and N13_FIGURE_DIR.exists():
    for pat in ["figure_smdg_reliability_pooled_raw_vs_scaled_primary_model.*",
                "figure_smdg_reliability_representative_sources_raw_vs_scaled_primary_model.*"]:
        for p in sorted(N13_FIGURE_DIR.glob(pat)):
            figure_candidates.append(("notebook13", p, "calibration"))

for fig_dir in N14_FIGURE_DIRS:
    if fig_dir and fig_dir.exists():
        for pat in ["notebook14_figure_hygd_quality_stratified_aqpr.*",
                    "notebook14_figure_smdg_source_shift_prevalence.*",
                    "notebook14_figure_hygd_forensic_duplicate_audit.*"]:
            for p in sorted(fig_dir.glob(pat)):
                figure_candidates.append(("notebook14", p, "quality_transport_forensics"))

if N15_FIGURE_DIR and N15_FIGURE_DIR.exists():
    for pat in ["notebook15_figure_saliency_sanity.*", "notebook15_figure_deletion*.*",
                "notebook15_figure_insertion*.*", "notebook15_figure_perturbation*.*"]:
        for p in sorted(N15_FIGURE_DIR.glob(pat)):
            figure_candidates.append(("notebook15", p, "expanded_core_audit"))

seen, fig_rows = set(), []
for src_nb, src, topic in figure_candidates:
    key = str(src.resolve())
    if key in seen:
        continue
    seen.add(key)
    copied = copy_if_exists(src, FIGURE_DIR, new_name=f"final_{src_nb}_{src.name}")
    role = "supplement"
    if src_nb == "notebook13" and "pooled" in src.name:
        role = "main_or_supplement_calibration"
    elif "smdg_source_shift" in src.name or "quality_stratified_aqpr" in src.name:
        role = "main_candidate"
    elif src_nb == "notebook15":
        role = "supplement_or_main_if_space"
    fig_rows.append({
        "figure_file": src.name,
        "source_notebook": src_nb,
        "topic": topic,
        "source_path": str(src),
        "copied_path": copied,
        "role_recommendation": role,
    })
figure_inventory = pd.DataFrame(fig_rows)
FIGURE_INVENTORY_CSV = FIGURE_DIR / "final_figure_inventory.csv"
figure_inventory.to_csv(FIGURE_INVENTORY_CSV, index=False)
print(figure_inventory.to_string(index=False) if not figure_inventory.empty else "No figures found.")


In [ ]:
# Cell 11: Reviewer response master matrix — v3.6

reviewer_master = pd.DataFrame([
    ["OD/OC proxy reliance without manual HYGD masks.", "addressed_with_guardrail", "Strict-valid proxy QC, proxy failure count, CDR distribution, and patient-level GON+/GON− CDR proxy difference.", "Report 745/747 strict-valid proxy masks and state that OD/OC masks are proxy audit regions, not manual HYGD ground truth.", "Manual HYGD OD/OC annotations remain unavailable."],
    ["Core sanity/deletion/perturbation endpoints used a small 60-image subset.", "resolved", "Notebook 15 expanded the core audit to all strict-valid HYGD images and 288 patients, with patient-level CIs.", "Replace old subset wording with all-HYGD expanded audit results and confidence intervals.", "Retrospective stress-test evidence, not prospective clinical trial evidence."],
    ["Limited uncertainty quantification for saliency/deletion/perturbation.", "resolved", "Bootstrap CIs, insertion/deletion AUC, OD/OC-minus-random perturbation CIs, and method-stability tables.", "Report CIs and AUC deltas in Results/Supplement.", "CIs quantify retrospective cohort variability."],
    ["Perturbation conclusions may depend on blur design.", "resolved_with_bounded_claim", "Blur, mean-fill, and Gaussian-noise perturbations across sigma/dilation settings.", "State method-dependent perturbation conclusions.", "Perturbations may induce distribution shift."],
    ["Calibration was pooled; per-source SMDG calibration was requested.", "resolved", "Per-source raw/scaled Brier, ECE-15, source eligibility, and reliability diagrams.", "Add source-level calibration table/figure and frozen temperature-scaling statement.", "No source-specific recalibration was performed."],
    ["Image-quality stratification lacks detail.", "resolved", "Quality-score source, tertile origin, thresholds, counts, and missingness.", "Define quality as preprocessing artifact used only for reliability stratification.", "Quality score is not a prospective clinical grading standard."],
    ["Patient-level aggregation may depend on unweighted mean pooling.", "resolved", "Mean, median, max, and quality-weighted patient aggregation tested.", "Retain unweighted mean as primary and report sensitivity in supplement.", "No learned patient pooling was trained."],
    ["SMDG label harmonization and source-level performance unclear.", "resolved", "Label policy, metric eligibility, source-level AUROC, and calibration diagnostics preserved.", "Report SMDG as source-level transportability audit.", "Some sources are single-class or metric-ineligible."],
    ["Overclaiming anatomical reliability.", "resolved_with_claim_boundary", "Expanded claim-boundary table distinguishes strongest bounded convergence from functionally mixed evidence.", "Use bounded claims for ConvNeXtTiny and EfficientNetB0.", "AQPR bounds reliability evidence; it does not prove clinical causality."],
], columns=["reviewer_concern", "revision_status", "evidence_generated", "manuscript_action", "residual_limitation"])

REVIEWER_MASTER_CSV = TABLE_DIR / "final_reviewer_response_master_matrix_v36.csv"
REVIEWER_MASTER_TEX = LATEX_DIR / "final_reviewer_response_master_matrix_v36.tex"
save_csv_md(reviewer_master, REVIEWER_MASTER_CSV)
write_latex_table(reviewer_master, REVIEWER_MASTER_TEX, "Reviewer concern, revision evidence, manuscript action, and residual limitation matrix.", "tab:reviewer_response_master_v36")
print(reviewer_master.to_string(index=False))


# Append v3.6 reviewer-specific concerns addressed by Notebooks 14 and 15.
reviewer_master_v36_extra = pd.DataFrame([
    ["Matched-random mask generator settings may affect enrichment/perturbation deltas.", "addressed_with_small_ablation", "Notebook 15 v3.6 varied K and maximum target overlap for saliency enrichment and perturbation-delta sensitivity.", "Report directional stability briefly and place full settings table in supplement.", "Sensitivity analysis is not a new primary endpoint."],
    ["AUC-delta definition for insertion/deletion is under-specified.", "resolved", "Notebook 15 v3.6 exports endpoint formula table and full curve points.", "Define trapezoidal AUC over fixed fractions and report full curves in supplement.", "Curve design remains an audit convention, not a clinical endpoint."],
    ["Attribution diversity is limited to IG.", "addressed_with_secondary_subset_check", "Notebook 15 v3.6 adds patch-occlusion robustness on a balanced 180-image HYGD subset.", "State that patch occlusion supports directional consistency but is secondary to all-HYGD IG audit.", "Not an exhaustive attribution-method benchmark."],
    ["Outside-OD high AUROC requires deeper forensic interpretation.", "addressed_with_forensic_association", "Notebook 14 v3.6 extracts non-OD cues such as corner-edge content, vignetting proxy, FOV area, saturation, and peripheral intensity statistics.", "Interpret as dataset-artifact/shortcut warning, not causal proof of model reliance.", "Does not isolate one causal shortcut mechanism."],
    ["Need proxy CDR/morphometric baseline versus end-to-end classifiers.", "addressed", "Notebook 14 v3.6 compares proxy CDR and shallow proxy morphometrics with frozen EfficientNetB0 patient-level HYGD summaries.", "Use as segmentation-first contextual baseline.", "Baseline contextualizes anatomy-derived information; it does not replace AQPR."],
], columns=["reviewer_concern", "revision_status", "evidence_generated", "manuscript_action", "residual_limitation"])

reviewer_master = pd.concat([reviewer_master, reviewer_master_v36_extra], ignore_index=True)
REVIEWER_MASTER_CSV = TABLE_DIR / "final_reviewer_response_master_matrix_v36.csv"
REVIEWER_MASTER_TEX = LATEX_DIR / "final_reviewer_response_master_matrix_v36.tex"
save_csv_md(reviewer_master, REVIEWER_MASTER_CSV)
write_latex_table(reviewer_master, REVIEWER_MASTER_TEX, "Reviewer concern, revision evidence, manuscript action, and residual limitation matrix.", "tab:reviewer_response_master_v36")
print("\nUpdated v3.6 reviewer response matrix:")
print(reviewer_master.to_string(index=False))


In [ ]:
# Cell 12: Final safe/unsafe manuscript claim-boundary audit — v3.6

claim_boundary = pd.DataFrame([
    ["Overall AQPR contribution", "AQPR audits anatomy-, quality-, patient-, calibration-, and source-level reliability and reveals convergent, partial, or divergent evidence.", "AQPR proves that high-AUROC classifiers reason clinically.", "Abstract, Introduction, Discussion"],
    ["OD/OC proxy", "HYGD anatomical analyses are based on externally developed OD/OC proxy masks with strict-valid QC and CDR face-validity checks.", "HYGD OD/OC masks are manual target-domain ground truth.", "Methods, Results, Limitations"],
    ["Quality stratification", "Quality was used as a preprocessing quality-score stratification variable for reliability analysis only.", "Quality tertiles are a prospective clinical image-quality standard.", "Methods and Limitations"],
    ["Patient aggregation", "Unweighted patient mean is primary, with median, max, and quality-weighted sensitivity analyses reported.", "Patient-level results are invariant to all aggregation choices.", "Methods, Results, Supplement"],
    ["EfficientNetB0", "EfficientNetB0 showed supported saliency sanity but mixed deletion/insertion evidence and replacement-dependent perturbation sensitivity.", "EfficientNetB0 is stably anatomically faithful.", "Results"],
    ["ConvNeXtTiny", "ConvNeXtTiny showed the strongest bounded AQPR profile, with supported deletion/insertion faithfulness and blur-supported but method-dependent OD/OC perturbation.", "ConvNeXtTiny universally relies on clinically plausible OD/OC anatomy.", "Results"],
    ["SMDG calibration", "Temperature scaling was fitted on AIROGS-light validation and applied frozen to SMDG; per-source ECE/Brier are transportability diagnostics.", "SMDG sources were recalibrated individually.", "Methods and Results"],
    ["Outside-OD controls", "Outside-OD controls are shortcut and dataset-artifact stress tests.", "Outside-OD results prove clinically meaningful non-anatomical causality.", "Results and Limitations"],
    ["Saliency maps", "Saliency maps are descriptive localization evidence interpreted with sanity, insertion/deletion, and perturbation controls.", "Saliency maps alone prove faithful clinical reasoning.", "Methods, Results, Limitations"],
], columns=["claim_topic", "safe_claim", "unsafe_claim", "paper_location"])

CLAIM_BOUNDARY_CSV = TABLE_DIR / "final_manuscript_claim_boundary_audit_v36.csv"
CLAIM_BOUNDARY_TEX = LATEX_DIR / "final_manuscript_claim_boundary_audit_v36.tex"
save_csv_md(claim_boundary, CLAIM_BOUNDARY_CSV)
write_latex_table(claim_boundary, CLAIM_BOUNDARY_TEX, "Safe and unsafe claims for manuscript v3.6.", "tab:claim_boundary_audit_v36")
print(claim_boundary.to_string(index=False))


# v3.6 additional claim boundaries.
claim_boundary_v36_extra = pd.DataFrame([
    ["Matched-random robustness", "K/overlap sensitivity checks supported directional stability of OD/OC enrichment and perturbation conclusions.", "Matched-random ablations prove invariance to all random-control designs.", "Methods, Supplement"],
    ["Patch-occlusion secondary attribution", "Patch occlusion on a balanced 180-image HYGD subset was used as secondary attribution-method sensitivity evidence.", "Patch occlusion replaces the all-HYGD IG primary audit.", "Results, Supplement"],
    ["Outside-OD forensic cues", "Non-OD image statistics were label-associated and support a shortcut/dataset-artifact warning.", "The analysis proves the classifier causally uses a specific artifact.", "Results, Limitations"],
    ["Proxy CDR baseline", "Proxy CDR/morphometrics provide a segmentation-first contextual baseline.", "CDR baseline replaces AQPR or fully explains end-to-end classifier behavior.", "Results, Discussion"],
], columns=["claim_topic", "safe_claim", "unsafe_claim", "paper_location"])

claim_boundary = pd.concat([claim_boundary, claim_boundary_v36_extra], ignore_index=True)
CLAIM_BOUNDARY_CSV = TABLE_DIR / "final_manuscript_claim_boundary_audit_v36.csv"
CLAIM_BOUNDARY_TEX = LATEX_DIR / "final_manuscript_claim_boundary_audit_v36.tex"
save_csv_md(claim_boundary, CLAIM_BOUNDARY_CSV)
write_latex_table(claim_boundary, CLAIM_BOUNDARY_TEX, "Safe and unsafe claims for manuscript v3.6.", "tab:claim_boundary_audit_v36")
print("\nUpdated v3.6 safe/unsafe claim-boundary audit:")
print(claim_boundary.to_string(index=False))


In [ ]:
# Cell 13: Manuscript-ready text fragments — v3.6

n14_handoff = safe_read_json(N14_HANDOFF_JSON)
n13_handoff = safe_read_json(N13_HANDOFF_JSON)
cdr_diff = safe_read_csv(N14_TABLE_DIR / "notebook14_hygd_proxy_patient_cdr_gon_difference_bootstrap.csv" if N14_TABLE_DIR else None)

proxy_n_valid = n14_handoff.get("proxy_quality", {}).get("n_strict_valid", 745)
proxy_n_fail = n14_handoff.get("proxy_quality", {}).get("n_proxy_failures", 2)
hygd_n = n14_handoff.get("hygd_manifest", {}).get("n_images", 747)

if not cdr_diff.empty:
    row = cdr_diff.iloc[0].to_dict()
    cdr_sentence = f"mean difference {float(row.get('mean_difference_positive_minus_negative')):.3f}; 95% CI {float(row.get('ci95_low')):.3f}-{float(row.get('ci95_high')):.3f}"
else:
    cdr_sentence = "mean difference and 95% CI reported in the supplement"

global_cal = n13_handoff.get("global_primary_calibration", {})
if global_cal.get("raw_ece_15", None) is not None and global_cal.get("scaled_ece_15", None) is not None:
    calibration_sentence = (
        f"For {PRIMARY_MODEL_NAME}, pooled SMDG ECE-15 changed from {float(global_cal['raw_ece_15']):.3f} "
        f"to {float(global_cal['scaled_ece_15']):.3f} after temperature scaling; "
        f"Brier score changed from {float(global_cal.get('raw_brier_score', np.nan)):.3f} "
        f"to {float(global_cal.get('scaled_brier_score', np.nan)):.3f}."
    )
else:
    calibration_sentence = "Per-source SMDG calibration diagnostics report raw and temperature-scaled Brier/ECE, with temperature fitted only on AIROGS-light validation."

fragments = {
    "methods_guardrails_v36.md": (
        f"The classifier panel was trained and selected using AIROGS-light internal development data only. "
        f"External datasets were not used for model training, model selection, threshold optimization, temperature fitting, "
        f"or source-specific recalibration. HYGD was used as an external patient-aware AQPR audit cohort. "
        f"HYGD anatomical localization used OD/OC proxy masks generated by an independently developed segmentation pipeline. "
        f"These masks passed strict-valid QC in {proxy_n_valid}/{hygd_n} images, but they are not manual HYGD OD/OC ground truth."
    ),
    "methods_quality_patient_aggregation_v36.md": (
        "HYGD quality stratification used the selected preprocessing quality-score artifact. Images were divided into low, "
        "medium, and high tertiles. Quality was used only for reliability stratification, not for classifier training, "
        "threshold selection, or calibration fitting. The primary patient-level endpoint used the unweighted mean of image-level "
        "probabilities; median, max, and quality-weighted mean aggregation were retained as supplementary robustness checks."
    ),
    "results_proxy_quality_v36.md": (
        f"The OD/OC proxy was strict-valid in {proxy_n_valid}/{hygd_n} HYGD images, with {proxy_n_fail} proxy-QC failures. "
        f"Among strict-valid images, the patient-level CDR proxy was higher in GON+ than in GON- patients ({cdr_sentence}). "
        "This supports use of the proxy as an anatomical audit region, while preserving the claim boundary that it is not manual HYGD segmentation ground truth."
    ),
    "results_architecture_dependent_aqpr_v36.md": (
        "The expanded all-HYGD audit showed architecture-dependent reliability patterns. ConvNeXtTiny had the strongest bounded "
        "AQPR profile: deletion/insertion faithfulness was supported and blur-based OD/OC perturbation was stable, although "
        "saliency-randomization and alternative perturbation controls require a bounded rather than universal anatomical-reliability claim. "
        "EfficientNetB0 showed supported saliency sanity but mixed deletion/insertion evidence and replacement-dependent perturbation sensitivity."
    ),
    "results_smdg_calibration_v36.md": (
        f"{calibration_sentence} Per-source calibration tables and reliability diagrams were used to connect calibration behavior "
        "to source-level domain shift. These diagnostics were interpreted as transportability evidence, not source-specific recalibration."
    ),
    "limitations_claim_boundaries_v36.md": (
        "HYGD OD/OC analyses rely on proxy masks from an external segmentation pipeline rather than manual HYGD OD/OC annotations. "
        "Perturbation and insertion/deletion tests are stress tests for functional dependence and may introduce distribution shift. "
        "SMDG showed source-level heterogeneity, so external validation should be interpreted as a transportability audit rather than uniform clinical generalization."
    ),
    "abstract_candidate_sentence_v36.md": (
        "AQPR revealed architecture-dependent reliability: ConvNeXtTiny showed the strongest bounded convergence across insertion/deletion "
        "faithfulness and blur-based OD/OC perturbation, whereas EfficientNetB0 showed saliency-supported but functionally mixed anatomical evidence."
    ),
}

rows = []
for name, content in fragments.items():
    p = TEXT_DIR / name
    p.write_text(content + "\\n", encoding="utf-8")
    rows.append({"fragment_name": name, "path": str(p), "n_words": len(content.split())})
FRAGMENT_INVENTORY_CSV = TEXT_DIR / "text_fragment_inventory_v36.csv"
pd.DataFrame(rows).to_csv(FRAGMENT_INVENTORY_CSV, index=False)
print(pd.DataFrame(rows).to_string(index=False))


# v3.6 reviewer-driven fragments from Notebook 14/15 handoffs.
v36_extra_fragments = {
    "results_matched_random_robustness_v36.md": (
        "Matched-random sensitivity checks varied the number of random masks and maximum target-overlap constraints. "
        "The direction of OD/OC saliency enrichment was preserved across settings, and perturbation-delta sensitivity was "
        "evaluated on a balanced subset. These analyses are reported as robustness checks, not as replacements for the all-HYGD primary endpoints."
    ),
    "results_secondary_patch_occlusion_v36.md": (
        "A secondary patch-occlusion attribution check on a balanced 180-image HYGD subset showed positive OD/OC enrichment "
        "and non-zero insertion/deletion deltas. This supports that the main IG-based audit was not solely an implementation artifact, "
        "while preserving IG as the primary all-HYGD attribution endpoint."
    ),
    "results_proxy_cdr_baseline_v36.md": (
        "Proxy CDR and shallow proxy-morphometric baselines provided meaningful segmentation-first anatomical baselines on HYGD. "
        "However, these baselines do not replace AQPR because AQPR also evaluates localization, functional dependence, calibration, "
        "quality effects, patient aggregation, and transportability."
    ),
    "results_outside_od_forensics_v36.md": (
        "Outside-OD forensic features, including corner-edge content, vignetting-related contrast, FOV area, saturation, and peripheral intensity statistics, "
        "were associated with HYGD labels. These associations support a dataset-artifact/shortcut warning but do not prove causal reliance on any single cue."
    ),
}

for fname, text in v36_extra_fragments.items():
    (TEXT_DIR / fname).write_text(text, encoding="utf-8")

# Preserve CSV/MD fragment table produced by Cell 9 if available.
print("\nAdded v3.6 manuscript-ready fragments:")
for fname in v36_extra_fragments:
    print(TEXT_DIR / fname)


In [ ]:
# Cell 14: Final v3.6 package manifest

items = []

def add_item(item_type, role, path, description):
    p = Path(path) if path else None
    items.append({
        "item_type": item_type,
        "role": role,
        "path": str(p) if p else "",
        "exists": bool(p.exists()) if p else False,
        "size_bytes": int(p.stat().st_size) if p and p.exists() else 0,
        "description": description,
    })

for p, desc in [
    (DATASET_ROLES_CSV, "Dataset roles and frozen component development."),
    (FINAL_QUALITY_CSV, "HYGD quality-stratified AQPR."),
    (PROXY_CDR_MAIN_CSV, "HYGD proxy CDR face-validity."),
    (FINAL_CLAIM_MAIN_CSV, "Expanded all-HYGD claim boundary."),
    (FINAL_CLAIM_NUMERIC_CSV, "Expanded claim numerical anchors."),
    (FINAL_SMDG_TRANSPORT_CSV, "SMDG source transportability."),
    (FINAL_SMDG_CAL_SOURCE_CSV, "SMDG source calibration."),
    (FINAL_SMDG_CAL_GLOBAL_CSV, "SMDG global calibration."),
    (FINAL_AGGREGATION_CSV, "Patient aggregation robustness."),
    (REVIEWER_MASTER_CSV, "Reviewer response matrix."),
    (CLAIM_BOUNDARY_CSV, "Safe/unsafe claim audit."),
]:
    add_item("table_csv", "main_or_supplement", p, desc)

for p in sorted(LATEX_DIR.glob("*.tex")):
    add_item("latex_table", "latex", p, p.name)
for p in sorted(SUPP_TABLE_DIR.glob("*.csv")):
    add_item("supplementary_csv", "supplement", p, p.name)
for p in sorted(FIGURE_DIR.glob("*")):
    if p.suffix.lower() in [".png", ".pdf", ".jpg", ".jpeg", ".svg"]:
        add_item("figure", "main_or_supplement", p, p.name)
for p in sorted(TEXT_DIR.glob("*.md")):
    add_item("text_fragment", "manuscript_revision", p, p.name)
for p in sorted(AUDIT_DIR.glob("*.csv")):
    add_item("audit_csv", "traceability", p, p.name)
for p in [N13_HANDOFF_JSON, N14_HANDOFF_JSON, N15_HANDOFF_JSON]:
    if p:
        add_item("handoff_json", "traceability", p, Path(p).name)

manifest = pd.DataFrame(items)
PACKAGE_MANIFEST_CSV = PACKAGE_DIR / "final_paper_revision_package_manifest_v36.csv"
ROOT_MANIFEST_CSV = RESULTS_16_DIR / "final_paper_revision_package_manifest_v36.csv"
manifest.to_csv(PACKAGE_MANIFEST_CSV, index=False)
manifest.to_csv(ROOT_MANIFEST_CSV, index=False)
print(manifest.to_string(index=False))


# Add v3.6 compact reviewer-driven outputs.
for p, desc in [
    (TABLE_DIR / "final_v36_table_matched_random_robustness_compact.csv", "v3.6 matched-random robustness compact table."),
    (TABLE_DIR / "final_v36_table_secondary_patch_occlusion_robustness.csv", "v3.6 secondary patch-occlusion robustness table."),
    (TABLE_DIR / "final_v36_table_proxy_morphometric_vs_end_to_end.csv", "v3.6 proxy morphometric baseline versus end-to-end table."),
    (TABLE_DIR / "final_v36_table_outside_od_root_cause_top_features.csv", "v3.6 outside-OD root-cause top features table."),
    (TABLE_DIR / "final_v36_table_endpoint_parameters.csv", "v3.6 endpoint parameter table."),
    (TABLE_DIR / "final_v36_table_auc_delta_formula.csv", "v3.6 AUC-delta formula table."),
]:
    add_item("table_csv", "main_or_supplement_v36", p, desc)

manifest = pd.DataFrame(items)
PACKAGE_MANIFEST_CSV = PACKAGE_DIR / "final_paper_revision_package_manifest_v36.csv"
ROOT_MANIFEST_CSV = RESULTS_16_DIR / "final_paper_revision_package_manifest_v36.csv"
manifest.to_csv(PACKAGE_MANIFEST_CSV, index=False)
manifest.to_csv(ROOT_MANIFEST_CSV, index=False)
print("\nUpdated v3.6 package manifest:")
print(manifest.to_string(index=False))


In [ ]:
# Cell 15: Final v3.6 readiness checklist

checks = [
    ("Notebook 13 per-source SMDG calibration available", [N13_HANDOFF_JSON, FINAL_SMDG_CAL_SOURCE_CSV]),
    ("Notebook 14 quality definition available", [N14_TABLE_DIR / "notebook14_hygd_quality_score_definition.csv" if N14_TABLE_DIR else None]),
    ("Notebook 14 proxy-QC and CDR evidence available", [N14_TABLE_DIR / "notebook14_hygd_proxy_quality_evidence_summary.csv" if N14_TABLE_DIR else None, PROXY_CDR_MAIN_CSV]),
    ("Notebook 14 patient aggregation robustness available", [N14_TABLE_DIR / "notebook14_hygd_patient_aggregation_robustness_compact.csv" if N14_TABLE_DIR else None, FINAL_AGGREGATION_CSV]),
    ("Notebook 15 expanded all-HYGD claim boundary available", [N15_TABLE_DIR / "notebook15_expanded_model_claim_boundary_full.csv" if N15_TABLE_DIR else None, FINAL_CLAIM_MAIN_CSV]),
    ("Notebook 15 expanded CIs preserved", [SUPP_TABLE_DIR / "S17_saliency_sanity_degradation_ci.csv", SUPP_TABLE_DIR / "S18_deletion_insertion_auc_guided_minus_random_ci.csv", SUPP_TABLE_DIR / "S19_perturbation_odoc_minus_random_ci.csv"]),
    ("SMDG label harmonization and source transportability preserved", [FINAL_SMDG_TRANSPORT_CSV, SUPP_TABLE_DIR / "S15_smdg_label_harmonization_policy.csv"]),
    ("HYGD duplicate/near-duplicate forensics preserved", [SUPP_TABLE_DIR / "S13_hygd_near_duplicate_threshold_summary.csv", SUPP_TABLE_DIR / "S14_hygd_exact_duplicate_groups.csv"]),
    ("Reviewer response matrix generated", [REVIEWER_MASTER_CSV]),
    ("Safe/unsafe claim-boundary audit generated", [CLAIM_BOUNDARY_CSV]),
    ("Manuscript text fragments generated", [TEXT_DIR / "methods_guardrails_v36.md", TEXT_DIR / "results_architecture_dependent_aqpr_v36.md"]),
    ("Figure inventory generated", [FIGURE_INVENTORY_CSV]),
]

rows = []
for requirement, paths in checks:
    clean_paths = [p for p in paths if p is not None]
    ready = all(exists_nonempty(p) for p in clean_paths)
    rows.append({
        "requirement": requirement,
        "status": "ready" if ready else "missing",
        "evidence": "; ".join(map(str, clean_paths)),
    })

readiness = pd.DataFrame(rows)
READINESS_CSV = RESULTS_16_DIR / "final_revision_readiness_checklist_v36.csv"
readiness.to_csv(READINESS_CSV, index=False)

final_summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 14",
    "status": "READY_FOR_MANUSCRIPT_V36_REVISION" if readiness["status"].eq("ready").all() else "READY_WITH_MISSING_ITEMS",
    "readiness_checklist_csv": str(READINESS_CSV),
    "package_manifest_csv": str(PACKAGE_MANIFEST_CSV),
    "n_requirements": int(len(readiness)),
    "n_ready": int(readiness["status"].eq("ready").sum()),
    "n_missing": int(readiness["status"].ne("ready").sum()),
    "next_step": "Use this package to revise the ICT Express manuscript into v3.6 minor-revision-ready.",
}
FINAL_SUMMARY_JSON = RESULTS_16_DIR / "notebook16_v36_final_execution_summary.json"
with open(FINAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)
FINAL_REPORT_TXT = RESULTS_16_DIR / "notebook16_v36_final_report.txt"
FINAL_REPORT_TXT.write_text("Notebook 16 v3.6 final report\\n" + json.dumps(final_summary, indent=2) + "\\n", encoding="utf-8")

print(readiness.to_string(index=False))
print("\\nFinal summary:")
print(json.dumps(final_summary, indent=2))


# v3.6 minor-revision readiness checks.
v36_checks = [
    ("Notebook 15 v3.6 endpoint parameter table available", [TABLE_DIR / "final_v36_table_endpoint_parameters.csv"]),
    ("Notebook 15 v3.6 AUC-delta formula available", [TABLE_DIR / "final_v36_table_auc_delta_formula.csv"]),
    ("Notebook 15 v3.6 matched-random robustness available", [TABLE_DIR / "final_v36_table_matched_random_robustness_compact.csv"]),
    ("Notebook 15 v3.6 secondary patch-occlusion robustness available", [TABLE_DIR / "final_v36_table_secondary_patch_occlusion_robustness.csv"]),
    ("Notebook 14 v3.6 proxy morphometric baseline available", [TABLE_DIR / "final_v36_table_proxy_morphometric_vs_end_to_end.csv"]),
    ("Notebook 14 v3.6 outside-OD root-cause forensics available", [TABLE_DIR / "final_v36_table_outside_od_root_cause_top_features.csv"]),
    ("Notebook 14 v3.6 reviewer handoff available", [N14_V36_HANDOFF_CSV]),
    ("Notebook 15 v3.6 reviewer handoff available", [N15_V36_HANDOFF_JSON]),
]

v36_rows = []
for requirement, paths in v36_checks:
    clean_paths = [p for p in paths if p is not None]
    ready = all(exists_nonempty(p) for p in clean_paths)
    v36_rows.append({
        "requirement": requirement,
        "status": "ready" if ready else "missing",
        "evidence": "; ".join(map(str, clean_paths)),
    })

readiness_v36_extra = pd.DataFrame(v36_rows)
readiness = pd.concat([readiness, readiness_v36_extra], ignore_index=True)
READINESS_CSV = RESULTS_16_DIR / "final_revision_readiness_checklist_v36.csv"
readiness.to_csv(READINESS_CSV, index=False)

final_summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 14",
    "status": "READY_FOR_MANUSCRIPT_V36_FINAL_MINOR_REVISION" if readiness["status"].eq("ready").all() else "READY_WITH_MISSING_ITEMS",
    "readiness_checklist_csv": str(READINESS_CSV),
    "package_manifest_csv": str(PACKAGE_MANIFEST_CSV),
    "n_requirements": int(len(readiness)),
    "n_ready": int(readiness["status"].eq("ready").sum()),
    "n_missing": int(readiness["status"].ne("ready").sum()),
    "next_step": "Use this package to revise the ICT Express manuscript into v3.6 final minor-revision-ready.",
}
FINAL_SUMMARY_JSON = RESULTS_16_DIR / "notebook16_v36_final_execution_summary.json"
with open(FINAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)
FINAL_REPORT_TXT = RESULTS_16_DIR / "notebook16_v36_final_report.txt"
FINAL_REPORT_TXT.write_text("Notebook 16 v3.6 final report\n" + json.dumps(final_summary, indent=2) + "\n", encoding="utf-8")

print("\nUpdated v3.6 readiness checklist:")
print(readiness.to_string(index=False))
print("\nFinal v3.6 summary:")
print(json.dumps(final_summary, indent=2))


In [ ]:
# Cell 16: Create a single ZIP package for the Notebook 16 v3.6 revision package

from pathlib import Path
import shutil
import json
from datetime import datetime

if not RESULTS_16_DIR.exists():
    raise FileNotFoundError(f"Notebook 16 v3.6 results folder not found: {RESULTS_16_DIR}")

ZIP_STEM = RESULTS_16_DIR.parent / "16_ict_express_revision_tables_figures_and_response_package_v36_FINAL"
ZIP_PATH = Path(str(ZIP_STEM) + ".zip")
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

created_zip = shutil.make_archive(
    base_name=str(ZIP_STEM),
    format="zip",
    root_dir=str(RESULTS_16_DIR.parent),
    base_dir=RESULTS_16_DIR.name,
)

summary = {
    "notebook": NOTEBOOK_NAME,
    "cell": "Cell 15",
    "status": "PASS",
    "results_16_dir": str(RESULTS_16_DIR),
    "zip_path": str(created_zip),
    "zip_exists": Path(created_zip).exists(),
    "zip_size_mb": round(Path(created_zip).stat().st_size / (1024 * 1024), 3) if Path(created_zip).exists() else 0,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "instruction": "Use this ZIP as the consolidated evidence package for manuscript v3.6 revision.",
}
with open(RESULTS_16_DIR / "notebook16_v36_cell15_zip_package_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Notebook 16 v3.6 ZIP package created:")
print(json.dumps(summary, indent=2))
